In [ ]:
from __future__ import annotations

import json, os, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from geometry import BACKBONE_LAYERS
import geometry
from model import SiamNCCLocalizer, _ConvBlock, _Backbone, BACKBONE_CHANNELS

In [2]:
BACKBONE_CHANNELS = [32, 64, 128, 128]


class _ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, s, p):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class _Backbone(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        layers = []
        c_in = in_channels
        for (k, s, p), c_out in zip(BACKBONE_LAYERS, BACKBONE_CHANNELS):
            layers.append(_ConvBlock(c_in, c_out, k, s, p))
            c_in = c_out
        self.net = nn.Sequential(*layers)
        self.out_channels = c_in

    def forward(self, x):
        return self.net(x)


class SiamNCCLocalizer(nn.Module):
    """
    forward(ref, search) -> (heatmap, offset)
        ref:     (B, 1, 100, 100)
        search:  (B, 1, 1000, 1000)
        heatmap: (B, 1, 226, 226)  raw logits
        offset:  (B, 2, 226, 226)  channel 0 = dx, channel 1 = dy
    """

    def __init__(self):
        super().__init__()
        self.backbone = _Backbone(in_channels=1)
        self.corr_scale = nn.Parameter(torch.tensor(10.0))
        self.corr_bias = nn.Parameter(torch.tensor(0.0))
        self.offset_head = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 2, kernel_size=1),
        )

    def _embed(self, x):
        feat = self.backbone(x)
        return F.normalize(feat, p=2, dim=1, eps=1e-6)
    
    def _xcorr(self, search_feat, ref_feat):
        """Depth-summed cross-correlation, computed per-sample with plain
        conv2d (groups=1) instead of one grouped conv with groups=batch_size.
        The grouped-conv-as-batch trick is faster in principle, but some
        cuDNN backend versions (notably on Windows) fail to find an engine
        for certain group counts -- most visibly at batch size 1, which is
        exactly the kind of batch a drop_last=False val_loader produces on
        its last iteration. Looping is marginally slower but never hits that
        failure mode."""
        B, C, Hs, Ws = search_feat.shape
        Br, Cr, Hr, Wr = ref_feat.shape
        assert B == Br and C == Cr, "reference/search batch or channel mismatch"

        outs = []
        for i in range(B):
            s = search_feat[i:i+1]          # (1, C, Hs, Ws)
            k = ref_feat[i:i+1]             # (1, C, Hr, Wr) used as a single kernel
            outs.append(F.conv2d(s, k))     # (1, 1, Ho, Wo)
        return torch.cat(outs, dim=0)       # (B, 1, Ho, Wo)

    def forward(self, ref, search):
        ref_feat = self._embed(ref)
        search_feat = self._embed(search)
        corr = self._xcorr(search_feat, ref_feat)
        heatmap = corr * self.corr_scale + self.corr_bias
        offset = self.offset_head(corr)
        return heatmap, offset

In [3]:
_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {_device}")

_model = SiamNCCLocalizer().to(_device)
_ref = torch.randn(2, 1, 100, 100, device=_device)
_search = torch.randn(2, 1, 1000, 1000, device=_device)

_heatmap, _offset = _model(_ref, _search)
print("heatmap:", _heatmap.shape)   # expect (2, 1, 226, 226)
print("offset:", _offset.shape)     # expect (2, 2, 226, 226)

_expected = geometry.valid_corr_size(1000, 100)
assert _heatmap.shape[-1] == _expected == _heatmap.shape[-2], \
    f"grid mismatch: model={_heatmap.shape[-2:]}, geometry expects {_expected}"
print("OK: model output grid matches geometry.valid_corr_size()")

del _model, _ref, _search, _heatmap, _offset  # free GPU mem before real training

device: cuda


NameError: name 'BACKBONE_LAYERS' is not defined

In [ ]:
class Args:
    # train
    data_dir = "output"
    epochs = 30
    batch_size = 8
    lr = 3e-4
    weight_decay = 1e-4
    lambda_offset = 1.0
    val_frac = 0.15
    num_workers = 4
    amp = True
    device = "cuda"
    out = "driftsense_v2_best.pt"
    log_every = 20
    resume = None

    # test
    test_dir = "output"
    checkpoint = "driftsense_v2_best.pt"
    debug = False
    save_report = None


args = Args()


In [ ]:
def _build_loaders(data_dir, batch_size, val_frac, num_workers):
    from dataset import WinspPairDataset

    full_ds = WinspPairDataset(data_dir)
    n_val = max(1, int(len(full_ds) * val_frac))
    n_train = len(full_ds) - n_val
    train_ds, val_ds = random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(0),
    )
    common = dict(
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
    )
    train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, **common)
    val_loader = DataLoader(val_ds, shuffle=False, drop_last=True, **common)
    return train_loader, val_loader


def _compute_loss(heatmap_pred, offset_pred, batch, device, lambda_offset, bce_fn, l1_fn):
    heatmap_gt = batch["heatmap"].to(device, non_blocking=True)
    offset_gt = batch["offset"].to(device, non_blocking=True)
    mask = batch["offset_mask"].to(device, non_blocking=True)
    in_grid = batch["target_in_grid"].to(device, non_blocking=True).float()

    per_sample_bce = bce_fn(heatmap_pred, heatmap_gt).mean(dim=(1, 2, 3))
    heatmap_loss = (per_sample_bce * in_grid).sum() / in_grid.sum().clamp(min=1.0)

    mask2 = mask.expand_as(offset_pred) * in_grid.view(-1, 1, 1, 1)
    diff = l1_fn(offset_pred, offset_gt)
    n_active = mask2.sum().clamp(min=1.0)
    offset_loss = (diff * mask2).sum() / n_active

    total = heatmap_loss + lambda_offset * offset_loss
    return total, heatmap_loss.detach(), offset_loss.detach()


@torch.no_grad()
def _validate(model, loader, device, lambda_offset, bce_fn, l1_fn):
    model.eval()
    tot, tot_hm, tot_off, n = 0.0, 0.0, 0.0, 0
    for batch in loader:
        ref = batch["reference"].to(device, non_blocking=True)
        search = batch["search"].to(device, non_blocking=True)
        heatmap_pred, offset_pred = model(ref, search)
        loss, hm_l, off_l = _compute_loss(
            heatmap_pred, offset_pred, batch, device, lambda_offset, bce_fn, l1_fn
        )
        tot += loss.item()
        tot_hm += hm_l.item()
        tot_off += off_l.item()
        n += 1
    model.train()
    n = max(n, 1)
    return tot / n, tot_hm / n, tot_off / n


def run_train(args):
    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        print("WARNING: CUDA not available, training on CPU will be very slow.")
    print(f"device: {device}")

    train_loader, val_loader = _build_loaders(
        args.data_dir, args.batch_size, args.val_frac, args.num_workers
    )
    print(f"train batches/epoch: {len(train_loader)}  val batches: {len(val_loader)}")

    model = SiamNCCLocalizer().to(device)
    if args.resume and os.path.exists(args.resume):
        model.load_state_dict(torch.load(args.resume, map_location=device))
        print(f"resumed from {args.resume}")

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)

    bce_fn = nn.BCEWithLogitsLoss(reduction="none")
    l1_fn = nn.SmoothL1Loss(reduction="none")
    scaler = torch.cuda.amp.GradScaler(enabled=args.amp and device.type == "cuda")

    best_val = float("inf")
    for epoch in range(1, args.epochs + 1):
        model.train()
        t0 = time.time()
        running = 0.0
        for step, batch in enumerate(train_loader, 1):
            ref = batch["reference"].to(device, non_blocking=True)
            search = batch["search"].to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=args.amp and device.type == "cuda"):
                heatmap_pred, offset_pred = model(ref, search)
                loss, hm_l, off_l = _compute_loss(
                    heatmap_pred, offset_pred, batch, device,
                    args.lambda_offset, bce_fn, l1_fn,
                )

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(opt)
            scaler.update()

            running += loss.item()
            if step % args.log_every == 0:
                elapsed = time.time() - t0
                ips = step * args.batch_size / elapsed
                print(f"epoch {epoch} step {step}/{len(train_loader)} "
                      f"loss={running/step:.4f} hm={hm_l.item():.4f} off={off_l.item():.4f} "
                      f"({ips:.1f} img/s)")

        sched.step()
        val_loss, val_hm, val_off = _validate(model, val_loader, device, args.lambda_offset, bce_fn, l1_fn)
        dt = time.time() - t0
        print(f"== epoch {epoch} done in {dt:.1f}s | train_loss={running/len(train_loader):.4f} "
              f"| val_loss={val_loss:.4f} (hm={val_hm:.4f} off={val_off:.4f}) | lr={sched.get_last_lr()[0]:.2e}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), args.out)
            print(f"  -> new best, saved to {args.out}")

    print(f"training complete. best val_loss={best_val:.4f}, weights at {args.out}")
    return model

In [ ]:
trained_model = run_train(args)

In [ ]:
def run_test(args):
    from inference import evaluate

    results, summary = evaluate(args.test_dir, args.checkpoint, args.device, debug=args.debug)

    print("\n=== per-pair results (worst 10 by error) ===")
    for r in sorted(results, key=lambda r: r["error_px"], reverse=True)[:10]:
        flag = " [HARD]" if r["hard_case"] else ""
        amb = f" ambiguous={r['n_distinct_ambiguous_matches']}" if r["n_distinct_ambiguous_matches"] > 1 else ""
        print(f"  pair {r['pair_id']:04d}  err={r['error_px']:6.2f}px  t={r['time_sec']*1000:6.1f}ms{flag}{amb}")

    print("\n=== summary ===")
    print(json.dumps(summary, indent=2))

    if args.save_report:
        with open(args.save_report, "w") as f:
            json.dump({"summary": summary, "results": results}, f, indent=2)
        print(f"\nfull report written to {args.save_report}")

    return results, summary


results, summary = run_test(args)